In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import re

In [3]:
train_data=pd.read_csv("Dataset/new_data_train.csv")
test_data=pd.read_csv("Dataset/new_data_test.csv")

print(train_data.head())
print(train_data.shape)
print(train_data.columns)
#print(test_data.head())


                                                text  label
0  Here are Thursday's biggest analyst calls: App...      0
1  Buy Las Vegas Sands as travel to Singapore bui...      0
2  Piper Sandler downgrades DocuSign to sell, cit...      0
3  Analysts react to Tesla's latest earnings, bre...      0
4  Netflix and its peers are set for a ‘return to...      0
(16990, 2)
Index(['text', 'label'], dtype='object')


In [5]:
print("=" * 70)
print("DATASET STRUCTURE")
print("=" * 70)

print(f"\nTraining set:")
print(f"  Total samples: {len(train_data)}")
print(f"  Columns: {list(train_data.columns)}")
print(f"  Data types: {train_data.dtypes.to_dict()}")

print(f"\nTest set:")
print(f"  Total samples: {len(test_data)}")
print(f"  Columns: {list(test_data.columns)}")

print(f"\nSample rows from training set:")
print(train_data.head(3))

print(f"\nMissing values:")
print(f"  Train - text: {train_data['text'].isna().sum()}")
print(f"  Train - label: {train_data['label'].isna().sum()}")
print(f"  Test - text: {test_data['text'].isna().sum()}")
print(f"  Test - label: {test_data['label'].isna().sum()}")

DATASET STRUCTURE

Training set:
  Total samples: 16990
  Columns: ['text', 'label']
  Data types: {'text': dtype('O'), 'label': dtype('int64')}

Test set:
  Total samples: 4117
  Columns: ['text', 'label']

Sample rows from training set:
                                                text  label
0  Here are Thursday's biggest analyst calls: App...      0
1  Buy Las Vegas Sands as travel to Singapore bui...      0
2  Piper Sandler downgrades DocuSign to sell, cit...      0

Missing values:
  Train - text: 0
  Train - label: 0
  Test - text: 0
  Test - label: 0


In [6]:
print("\n" + "=" * 70)
print("TEXT LENGTH ANALYSIS")
print("=" * 70)

# Calculate text lengths
train_data['text_length'] = train_data['text'].str.len()
train_data['token_count'] = train_data['text'].str.split().str.len()

test_data['text_length'] = test_data['text'].str.len()
test_data['token_count'] = test_data['text'].str.split().str.len()

print(f"\nCharacter length statistics:")
print(f"  Train set:")
print(f"    Min: {train_data['text_length'].min()} chars")
print(f"    Max: {train_data['text_length'].max()} chars")
print(f"    Mean: {train_data['text_length'].mean():.0f} chars")
print(f"    Median: {train_data['text_length'].median():.0f} chars")
print(f"  Test set:")
print(f"    Min: {test_data['text_length'].min()} chars")
print(f"    Max: {test_data['text_length'].max()} chars")
print(f"    Mean: {test_data['text_length'].mean():.0f} chars")
print(f"    Median: {test_data['text_length'].median():.0f} chars")

print(f"\nToken count statistics:")
print(f"  Train set:")
print(f"    Min: {train_data['token_count'].min()} tokens")
print(f"    Max: {train_data['token_count'].max()} tokens")
print(f"    Mean: {train_data['token_count'].mean():.1f} tokens")
print(f"    Median: {train_data['token_count'].median():.0f} tokens")
print(f"  Test set:")
print(f"    Min: {test_data['token_count'].min()} tokens")
print(f"    Max: {test_data['token_count'].max()} tokens")
print(f"    Mean: {test_data['token_count'].mean():.1f} tokens")
print(f"    Median: {test_data['token_count'].median():.0f} tokens")

print(f"\nToken count percentiles (training set):")
percentiles = [25, 50, 75, 90, 95, 99]
for p in percentiles:
    val = np.percentile(train_data['token_count'], p)
    print(f"  {p}th percentile: {val:.0f} tokens")

# How much padding will we need?
current_padding = 64
median_tokens = train_data['token_count'].median()
p95_tokens = np.percentile(train_data['token_count'], 95)

print(f"\nPadding efficiency:")
print(f"  Current padding length: 64 tokens")
print(f"  Median text: {median_tokens:.0f} tokens (padding waste: {64-median_tokens:.0f})")
print(f"  95th percentile: {p95_tokens:.0f} tokens")
if p95_tokens > 64:
    print(f"  ⚠️ WARNING: 95th percentile exceeds 64! Some text will be truncated!")
else:
    print(f"  ✓ 64 tokens covers 95% of data")


TEXT LENGTH ANALYSIS

Character length statistics:
  Train set:
    Min: 2 chars
    Max: 316 chars
    Mean: 136 chars
    Median: 126 chars
  Test set:
    Min: 5 chars
    Max: 308 chars
    Mean: 136 chars
    Median: 125 chars

Token count statistics:
  Train set:
    Min: 1 tokens
    Max: 64 tokens
    Mean: 18.3 tokens
    Median: 16 tokens
  Test set:
    Min: 1 tokens
    Max: 53 tokens
    Mean: 18.4 tokens
    Median: 16 tokens

Token count percentiles (training set):
  25th percentile: 12 tokens
  50th percentile: 16 tokens
  75th percentile: 23 tokens
  90th percentile: 31 tokens
  95th percentile: 36 tokens
  99th percentile: 44 tokens

Padding efficiency:
  Current padding length: 64 tokens
  Median text: 16 tokens (padding waste: 48)
  95th percentile: 36 tokens
  ✓ 64 tokens covers 95% of data


In [7]:
print("\n" + "=" * 70)
print("CLASS DISTRIBUTION")
print("=" * 70)

# Count labels
train_label_dist = train_data['label'].value_counts().sort_index()
test_label_dist = test_data['label'].value_counts().sort_index()

print(f"\nTraining set distribution:")
print(f"  Total samples: {len(train_data)}")
print(f"  Unique classes: {train_data['label'].nunique()}")

print(f"\nTest set distribution:")
print(f"  Total samples: {len(test_data)}")
print(f"  Unique classes: {test_data['label'].nunique()}")

print(f"\nDetailed breakdown:")
print(f"\n{'Class':<8} {'Train Count':<15} {'Train %':<12} {'Test Count':<15} {'Test %':<12} {'Stratified?':<15}")
print("-" * 80)

all_match = True
for class_idx in range(20):
    train_count = train_label_dist.get(class_idx, 0)
    test_count = test_label_dist.get(class_idx, 0)
    train_pct = train_count / len(train_data) * 100 if train_count > 0 else 0
    test_pct = test_count / len(test_data) * 100 if test_count > 0 else 0

    # Check if proportions match (within 2%)
    if train_count > 0 and test_count > 0:
        stratified = "✓ OK" if abs(train_pct - test_pct) < 2 else "⚠️ OFF"
        if abs(train_pct - test_pct) >= 2:
            all_match = False
    else:
        stratified = "❌ MISSING"
        all_match = False
    
    print(f"{class_idx:<8} {train_count:<15} {train_pct:<12.2f} {test_count:<15} {test_pct:<12.2f} {stratified:<15}")

print(f"\nImbalance assessment:")
min_train = train_label_dist.min()
max_train = train_label_dist.max()
imbalance_ratio = max_train / min_train

print(f"  Min class size: {min_train} samples")
print(f"  Max class size: {max_train} samples")
print(f"  Imbalance ratio: {imbalance_ratio:.1f}x")

if all_match:
    print(f"\n✓ Test set appears to be STRATIFIED (good!)")
else:
    print(f"\n⚠️ Test set may NOT be stratified (distribution differs from train)")


CLASS DISTRIBUTION

Training set distribution:
  Total samples: 16990
  Unique classes: 20

Test set distribution:
  Total samples: 4117
  Unique classes: 20

Detailed breakdown:

Class    Train Count     Train %      Test Count      Test %       Stratified?    
--------------------------------------------------------------------------------
0        255             1.50         73              1.77         ✓ OK           
1        837             4.93         214             5.20         ✓ OK           
2        3545            20.87        852             20.69        ✓ OK           
3        321             1.89         77              1.87         ✓ OK           
4        359             2.11         97              2.36         ✓ OK           
5        987             5.81         242             5.88         ✓ OK           
6        524             3.08         146             3.55         ✓ OK           
7        624             3.67         160             3.89         ✓ OK   

In [8]:
print("\n" + "=" * 70)
print("VOCABULARY ANALYSIS")
print("=" * 70)

# Tokenize all text (simple split, lowercase)
def tokenize(text):
    """Simple tokenization"""
    if pd.isna(text):
        return []
    return text.lower().split()

# Get all tokens from training set
all_tokens = []
for text in train_data['text']:
    tokens = tokenize(text)
    all_tokens.extend(tokens)

# Count token frequencies
token_counts = Counter(all_tokens)
unique_tokens = len(token_counts)

print(f"\nVocabulary statistics:")
print(f"  Total tokens (with repetition): {len(all_tokens):,}")
print(f"  Unique tokens: {unique_tokens:,}")
print(f"  Average tokens per sample: {len(all_tokens) / len(train_data):.1f}")

# Frequency distribution
frequencies = list(token_counts.values())

print(f"\nToken frequency distribution:")
print(f"  Max frequency: {max(frequencies):,} (most common word)")
print(f"  Min frequency: {min(frequencies)} (least common word)")
print(f"  Mean frequency: {np.mean(frequencies):.2f}")
print(f"  Median frequency: {np.median(frequencies):.1f}")

# How many tokens appear only X times?
thresholds = [1, 2, 3, 5, 10, 20, 50]
print(f"\nToken frequency thresholds:")
for threshold in thresholds:
    count = sum(1 for f in frequencies if f <= threshold)
    pct = count / unique_tokens * 100
    tokens_covered = sum(f for token, f in token_counts.items() if f <= threshold)
    tokens_covered_pct = tokens_covered / len(all_tokens) * 100
    
    print(f"  ≤{threshold:2d} occurrences: {count:6,} tokens ({pct:5.1f}%) → covers {tokens_covered_pct:5.1f}% of all tokens")

# Recommendation
print(f"\nVocabulary pruning recommendation:")
keep_tokens = sum(1 for f in frequencies if f >= 5)
remove_tokens = sum(1 for f in frequencies if f < 5)
tokens_kept_pct = sum(f for token, f in token_counts.items() if f >= 5) / len(all_tokens) * 100

print(f"  Current vocabulary: {unique_tokens:,} tokens")
print(f"  If threshold=5: Keep {keep_tokens:,} tokens, remove {remove_tokens:,} tokens")
print(f"  This covers {tokens_kept_pct:.1f}% of all tokens")
print(f"  Reduction: {(1 - keep_tokens/unique_tokens)*100:.1f}%")

print(f"\n  If threshold=10: Keep {sum(1 for f in frequencies if f >= 10):,} tokens")
print(f"  If threshold=20: Keep {sum(1 for f in frequencies if f >= 20):,} tokens")


VOCABULARY ANALYSIS

Vocabulary statistics:
  Total tokens (with repetition): 311,113
  Unique tokens: 55,496
  Average tokens per sample: 18.3

Token frequency distribution:
  Max frequency: 9,304 (most common word)
  Min frequency: 1 (least common word)
  Mean frequency: 5.61
  Median frequency: 1.0

Token frequency thresholds:
  ≤ 1 occurrences: 39,171 tokens ( 70.6%) → covers  12.6% of all tokens
  ≤ 2 occurrences: 44,791 tokens ( 80.7%) → covers  16.2% of all tokens
  ≤ 3 occurrences: 47,367 tokens ( 85.4%) → covers  18.7% of all tokens
  ≤ 5 occurrences: 49,813 tokens ( 89.8%) → covers  22.1% of all tokens
  ≤10 occurrences: 52,196 tokens ( 94.1%) → covers  27.9% of all tokens
  ≤20 occurrences: 53,634 tokens ( 96.6%) → covers  34.7% of all tokens
  ≤50 occurrences: 54,691 tokens ( 98.5%) → covers  45.3% of all tokens

Vocabulary pruning recommendation:
  Current vocabulary: 55,496 tokens
  If threshold=5: Keep 6,621 tokens, remove 48,875 tokens
  This covers 79.4% of all tokens

In [9]:
print("\n" + "=" * 70)
print("TOP 30 MOST FREQUENT WORDS")
print("=" * 70)

# Get top 30 tokens
top_30 = token_counts.most_common(30)

print(f"\n{'Rank':<6} {'Word':<25} {'Frequency':<12} {'% of corpus':<12}")
print("-" * 60)

total_tokens = len(all_tokens)
for rank, (word, count) in enumerate(top_30, 1):
    pct = count / total_tokens * 100
    print(f"{rank:<6} {word:<25} {count:<12,} {pct:<12.3f}%")

print(f"\nThese top 30 words account for {sum(c for _, c in top_30)/total_tokens*100:.1f}% of all tokens")

# Quality check
print("\n" + "=" * 70)
print("DATA QUALITY CHECK")
print("=" * 70)

# Check for duplicates
print(f"\nDuplicate detection:")
train_texts = train_data['text'].tolist()
unique_texts = len(set(train_texts))
print(f"  Total samples: {len(train_texts)}")
print(f"  Unique texts: {unique_texts}")
print(f"  Duplicates: {len(train_texts) - unique_texts}")

if len(train_texts) != unique_texts:
    print(f"  ⚠️ WARNING: Found {len(train_texts) - unique_texts} duplicate entries!")

# Check for very short texts
very_short = (train_data['token_count'] < 3).sum()
print(f"\nVery short texts (< 3 tokens):")
print(f"  Count: {very_short}")
print(f"  Percentage: {very_short/len(train_data)*100:.2f}%")

if very_short > 10:
    print(f"  ⚠️ WARNING: Many very short texts might be noise")

# Sample some very short ones
if very_short > 0:
    print(f"\n  Examples of very short texts:")
    short_samples = train_data[train_data['token_count'] < 3]['text'].head(5)
    for i, text in enumerate(short_samples, 1):
        print(f"    {i}. {text[:80]}")


TOP 30 MOST FREQUENT WORDS

Rank   Word                      Frequency    % of corpus 
------------------------------------------------------------
1      the                       9,304        2.991       %
2      to                        8,121        2.610       %
3      of                        5,797        1.863       %
4      a                         4,942        1.588       %
5      in                        4,755        1.528       %
6      -                         4,646        1.493       %
7      and                       3,913        1.258       %
8      for                       3,221        1.035       %
9      on                        2,762        0.888       %
10     is                        2,099        0.675       %
11     as                        2,044        0.657       %
12     with                      1,784        0.573       %
13     by                        1,475        0.474       %
14     its                       1,285        0.413       %
15     from

In [10]:
print("\n" + "=" * 70)
print("TEXT COMPOSITION ANALYSIS")
print("=" * 70)

# Check for URLs, special chars, etc.
urls = train_data['text'].str.contains('http', case=False).sum()
mentions = train_data['text'].str.contains('@', case=False).sum()
hashtags = train_data['text'].str.contains('#', case=False).sum()
numbers = train_data['text'].str.contains(r'\d', regex=True).sum()

print(f"\nSpecial content:")
print(f"  Samples with URLs: {urls} ({urls/len(train_data)*100:.1f}%)")
print(f"  Samples with @mentions: {mentions} ({mentions/len(train_data)*100:.1f}%)")
print(f"  Samples with #hashtags: {hashtags} ({hashtags/len(train_data)*100:.1f}%)")
print(f"  Samples with numbers: {numbers} ({numbers/len(train_data)*100:.1f}%)")

# Check character encoding issues
print(f"\nEncoding check:")
try:
    # Try to detect non-ASCII characters
    non_ascii = 0
    for text in train_data['text']:
        if not text.isascii():
            non_ascii += 1
    print(f"  Samples with non-ASCII characters: {non_ascii} ({non_ascii/len(train_data)*100:.2f}%)")
    if non_ascii == 0:
        print(f"  ✓ All text is ASCII/English")
except:
    print(f"  (Could not check encoding)")

# Sample texts from different classes
print(f"\n" + "=" * 70)
print("SAMPLE TEXTS FROM EACH CLASS")
print("=" * 70)

for class_idx in range(20):
    samples = train_data[train_data['label'] == class_idx]['text'].head(1)
    if len(samples) > 0:
        text = samples.iloc[0]
        tokens = len(text.split())
        print(f"\nClass {class_idx} (sample):")
        print(f"  Length: {tokens} tokens")
        print(f"  Text: {text[:120]}...")


TEXT COMPOSITION ANALYSIS

Special content:
  Samples with URLs: 15067 (88.7%)
  Samples with @mentions: 949 (5.6%)
  Samples with #hashtags: 1708 (10.1%)
  Samples with numbers: 14902 (87.7%)

Encoding check:
  Samples with non-ASCII characters: 3074 (18.09%)

SAMPLE TEXTS FROM EACH CLASS

Class 0 (sample):
  Length: 15 tokens
  Text: Here are Thursday's biggest analyst calls: Apple, Amazon, Tesla, Palantir, DocuSign, Exxon &amp; more  https://t.co/QPN8...

Class 1 (sample):
  Length: 20 tokens
  Text: LIVE: ECB surprises with 50bps hike, ending its negative rate era.   President Christine Lagarde is taking questions ▶️ ...

Class 2 (sample):
  Length: 36 tokens
  Text: Goldman Sachs traders countered the industry’s underwriting slump with revenue gains that raced past analysts’ estimates...

Class 3 (sample):
  Length: 33 tokens
  Text: China Evergrande Group’s onshore bond holders rejected a plan by the distressed developer to further extend a bond payme...

Class 4 (sample):
  Len

## Data Preprocessing

In [11]:
def clean_text(text):
    """
    Clean financial news text for training
    """
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower()
    
    # Remove URLs (they're just noise)
    text = re.sub(r'http\S+|www\S+', '', text)
    
    # Remove @mentions
    text = re.sub(r'@\w+', '', text)
    
    # Remove hashtags (but keep the word)
    text = re.sub(r'#(\w+)', r'\1', text)
    
    # Remove emojis and non-ASCII (financial text should be ASCII)
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Remove excessive punctuation (keep some like -, $, %)
    text = re.sub(r'[^\w\s\-$%]', ' ', text)
    
    # Remove extra whitespace
    text = ' '.join(text.split())
    
    return text

# Apply cleaning
train_data['text_clean'] = train_data['text'].apply(clean_text)
test_data['text_clean'] = test_data['text'].apply(clean_text)

# Check results
print(f"Before cleaning: {len(train_data['text'].iloc[0])} chars")
print(f"After cleaning: {len(train_data['text_clean'].iloc[0])} chars")
print(f"Sample: {train_data['text_clean'].iloc[0][:100]}")

Before cleaning: 126 chars
After cleaning: 93 chars
Sample: here are thursday s biggest analyst calls apple amazon tesla palantir docusign exxon amp more


In [13]:
# Remove texts with <5 tokens
train_data['token_count'] = train_data['text_clean'].str.split().str.len()
test_data['token_count'] = test_data['text_clean'].str.split().str.len()

print(f"Before filtering: {len(train_data)} training samples")

train_data = train_data[train_data['token_count'] >= 5].reset_index(drop=True)
test_data = test_data[test_data['token_count'] >= 5].reset_index(drop=True)

print(f"After filtering: {len(train_data)} training samples")
print(f"Removed: {16990 - len(train_data)} samples")

Before filtering: 16990 training samples
After filtering: 16612 training samples
Removed: 378 samples


In [14]:
# Tokenize all training text
all_tokens = []
for text in train_data['text_clean']:
    tokens = text.split()
    all_tokens.extend(tokens)

# Count frequencies
token_counts = Counter(all_tokens)

# Keep only tokens appearing ≥6 times
min_frequency = 6
vocab = {word: idx + 1 for idx, (word, count) in enumerate(
    sorted(token_counts.items(), key=lambda x: x[1], reverse=True)
    if count >= min_frequency
)}

# Add special tokens
vocab['<UNK>'] = 0  # Unknown token for OOV words

print(f"Original vocabulary: {len(token_counts):,} tokens")
print(f"Pruned vocabulary: {len(vocab):,} tokens")
print(f"Reduction: {(1 - len(vocab)/len(token_counts))*100:.1f}%")
print(f"Embedding matrix size: {len(vocab)} × 64 = {len(vocab)*64:,} params")

SyntaxError: expected 'else' after 'if' expression (2395522545.py, line 13)